In [2]:
# !modal volume create prj-vol
# !modal volume create data-vol
# !modal volume create models-vol

# !modal secret create general WANDB_API_KEY=13ad03bbf5464ebb2b5f4532464556bcc17811a2 HUGGINGFACEHUB_API_TOKEN=REMOVEDRpSsYkbcJiDfMtdEIyrVZPSQlKWKJooDBL


In [2]:
!pip install torchcodec==0.6.0 datasets neucodec -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
def setup_environment() -> None:
    import os
    from dotenv import load_dotenv
    _ = load_dotenv()

    from huggingface_hub import login
    login(token=os.environ['HUGGINGFACEHUB_API_TOKEN'])
   
setup_environment()

In [3]:
from datasets import load_from_disk, load_dataset

In [5]:
# ds = load_dataset('thivux/phoaudiobook')

In [6]:
# ds.save_to_disk('/mnt/data-vol/thivux__phoaudiobook')

In [13]:
%%writefile prepare_audio_codec.py
from datasets import load_dataset, load_from_disk

def get_audio_codes(example, model):
    audio = example["audio"]["array"]

    # (T,) -> (1, 1, T)
    y = torch.from_numpy(audio).float().unsqueeze(0).unsqueeze(0).cuda()

    with torch.no_grad():
        codes = model.encode_code(y)

    return {
        "codes": codes.flatten().cpu().tolist()
    }

import torch

def get_audio_codes_batched(examples, model):
    """
    Encode từng audio riêng lẻ, rồi append codes vào list
    """

    all_codes = []

    for audio in examples["audio"]:
        # (T,) -> (1, 1, T)
        y = (
            torch.from_numpy(audio["array"])
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
            # .cuda()
        )

        with torch.no_grad():
            codes = model.encode_code(y)

        # remove batch dim, flatten
        codes = codes.squeeze(0).flatten().cpu().tolist()
        all_codes.append(codes)

    return {
        "codes": all_codes
    }


def add_sample_id(dataset, start=0, prefix=None, width=0, column_name="sid"):
    """
    Add sample_id column to a HuggingFace Dataset.

    dataset     : datasets.Dataset
    start       : starting id
    prefix      : optional prefix (e.g. 's')
    width       : zero-padding width
    column_name : name of id column (default 'sid')
    """
    def format_sid(i):
        if width > 0:
            num = f"{i:0{width}d}"
        else:
            num = str(i)

        return f"{prefix}{num}" if prefix else num

    sids = [format_sid(i) for i in range(start, start + len(dataset))]
    return dataset.add_column(column_name, sids)




if __name__ == "__main__":

    from neucodec import NeuCodec
    from math import ceil
    import os

    model = NeuCodec.from_pretrained("neuphonic/neucodec")
    model.eval().cuda()
    model.eval()

    # dataset_name = "vivos"
    # dataset = load_dataset(dataset_name)

    dataset_name = "/mnt/data-vol/thivux__phoaudiobook"
    ds = load_from_disk(dataset_name)

    
    print("Adding sample_id...")
    ds["train"] = add_sample_id(ds["train"])
    ds["validation"] = add_sample_id(ds["validation"])
    ds["test"] = add_sample_id(ds["test"])

    print("Compute audio codes...")
    dataset = ds['train']
    
    chunk_size = 1_000
    num_samples = len(dataset)
    num_chunks = ceil(num_samples / chunk_size)

    
    LOCK_DIR = "/mnt/data-vol/phoaudiobook/interim/audio_codes/locks"
    OUT_DIR = "/mnt/data-vol/phoaudiobook/interim/audio_codes/codes"

    # os.makedirs('/mnt/data-vol/audio_codes', exist_ok=True)
    
    os.makedirs(LOCK_DIR, exist_ok=True)
    os.makedirs(OUT_DIR, exist_ok=True)


    
    # model = load_model().to("cuda")

    chunk_range = range(80, num_samples)
    
    for chunk_idx in chunk_range:
        start = chunk_idx * chunk_size
        end = min(start + chunk_size, num_samples)
    
        lock_path = f"{LOCK_DIR}/chunk_{start}_{end-1}.lock"
        out_path = f"{OUT_DIR}/train_neucodec_{start}_{end-1}"
    
        # 이미 xử lý xong
        if os.path.exists(out_path):
            continue
    
        # try to claim chunk
        try:
            os.mkdir(lock_path)
        except FileExistsError:
            # chunk đang bị script khác xử lý
            continue
    
        print(f"Processing chunk {start}-{end-1}")
    
        try:
            ds_chunk = dataset.select(range(start, end))
    
            ds_chunk = ds_chunk.map(
                get_audio_codes,
                fn_kwargs={"model": model},
                # batched=True,
                # batch_size=64,
                desc=f"Chunk {start}-{end-1}",
            )
    
            ds_chunk.save_to_disk(out_path)
    
        except Exception as e:
            print(f"Error in chunk {start}-{end-1}: {e}")
            # nếu fail → xoá lock để script khác retry
            os.rmdir(lock_path)
            raise
    
        # done → giữ lock lại như dấu đã xong (hoặc xoá cũng được)



Overwriting prepare_audio_codec.py


In [11]:
# python prepare_audio_codec.py &
# python prepare_audio_codec.py &
# python prepare_audio_codec.py &
# python prepare_audio_codec.py &
# python prepare_audio_codec.py

In [ ]:
python prepare_audio_codec.py &
python prepare_audio_codec.py &
python prepare_audio_codec.py &
python prepare_audio_codec.py &
python prepare_audio_codec.py